In [3]:
import pandas as pd
import glob
import os

# Folder path you provided
folder = "../weather_data"

# Collect all CSVs in the folder
filepaths = glob.glob(os.path.join(folder, "*.csv"))

dfs = []

for fp in filepaths:
    # --- 1. Read CSV, skipping the first 2 rows ---
    df = pd.read_csv(fp, skiprows=2)

    # Clean up column names (optional but nice)
    df.columns = df.columns.str.strip()

    # --- 2. Build datetime from Year/Month/Day/Hour/Minute ---
    df["datetime"] = pd.to_datetime(
        df[["Year", "Month", "Day", "Hour", "Minute"]],
        errors="coerce"
    )

    # --- 3. Floor to the hour (00–59 min → same hour) ---
    df["datetime_hour"] = df["datetime"].dt.floor("h")

    # --- 4. Localize to UTC (no conversion, timestamps are already UTC) ---
    df["datetime_utc"] = df["datetime_hour"].dt.tz_localize("UTC")

    dfs.append(df)

# --- 5. Combine all files ---
full_df = pd.concat(dfs, ignore_index=True)

# --- 6. Group by hourly UTC timestamp and average numeric columns ---
avg_df = (
    full_df
    .groupby("datetime_utc", as_index=False)
    .mean(numeric_only=True)
)

print(avg_df.head())

# Optional: save output
avg_df.to_csv("../data/weather_hourly_avg.csv", index=False)


               datetime_utc    Year  Month  Day  Hour  Minute  Temperature  \
0 2024-01-01 00:00:00+00:00  2024.0    1.0  1.0   0.0    30.0    11.141935   
1 2024-01-01 01:00:00+00:00  2024.0    1.0  1.0   1.0    30.0    10.087097   
2 2024-01-01 02:00:00+00:00  2024.0    1.0  1.0   2.0    30.0     9.457796   
3 2024-01-01 03:00:00+00:00  2024.0    1.0  1.0   3.0    30.0     9.013441   
4 2024-01-01 04:00:00+00:00  2024.0    1.0  1.0   4.0    30.0     8.747312   

      Alpha  Aerosol Optical Depth  Asymmetry  ...     Ozone  \
0  1.216156               0.035677   0.631559  ...  0.297852   
1  1.221720               0.039417   0.631559  ...  0.301024   
2  1.217312               0.043780   0.631559  ...  0.304624   
3  1.200269               0.048726   0.631559  ...  0.309624   
4  1.173710               0.052882   0.631559  ...  0.316159   

   Relative Humidity  Solar Zenith Angle       SSA  Surface Albedo  \
0          75.251183           86.770941  0.924462        0.140887   
1     